<a href="https://colab.research.google.com/github/narame7/UOS-FootballDataAnalytics-Tutorial/blob/main/Week%202/1-load-statistic-data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 통계 데이터 크롤링

In [ ]:
!pip install soccerdata

In [ ]:
import io
import re

import pandas as pd
import requests
from bs4 import BeautifulSoup
import soccerdata as sd

## 필요한 라이브러리 import

## [ClubElo](http://clubelo.com/)
- 전 세계 클럽의 Elo 점수(팀 전력 지수)를 계산해 순위를 매기는 사이트.
- 원래는 `api.clubelo.com` API가 있었지만 2026년 9월 현재 서버가 응답하지 않아(502), 웹페이지의 HTML 표를 `pandas.read_html`로 직접 읽어 옵니다.
- `read_html`은 페이지 안의 모든 `<table>`을 DataFrame 목록으로 돌려주므로, 원하는 표를 골라내는 과정이 필요합니다.

In [ ]:
import time

# 많은 사이트가 브라우저가 아닌 요청(파이썬 기본 User-Agent)을 차단하므로 브라우저처럼 보이는 헤더를 붙입니다.
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/128.0 Safari/537.36'}

def get_html(url, retries=3):
    """웹페이지 HTML을 받아옵니다. 서버가 연결을 끊거나 오류를 내면 잠시 쉬었다가 다시 시도합니다."""
    for attempt in range(retries):
        try:
            response = requests.get(url, headers=HEADERS, timeout=30)
            response.raise_for_status()  # 200이 아니면 오류
            return response.text
        except requests.RequestException as e:
            if attempt == retries - 1:
                raise
            print(f'요청 실패({e.__class__.__name__}), 5초 후 재시도합니다...')
            time.sleep(5)

def read_tables(url):
    """웹페이지 안의 모든 표를 DataFrame 목록으로 돌려줍니다."""
    return pd.read_html(io.StringIO(get_html(url)))

### 리그별 현재 Elo 순위
- 순위 페이지에는 리그마다 `Club`, `Elo` 두 열짜리 표가 하나씩 있습니다. `Club` 열은 `'2 ARSArsenal'`처럼 세계 순위 + 팀 코드 + 팀 이름이 붙어 있어 정규식으로 나눕니다.

In [ ]:
elo_pages = read_tables('http://clubelo.com/Ranking')
elo_tables = [t for t in elo_pages if list(t.columns) == ['Club', 'Elo']]
print(len(elo_pages), '개 표 중 리그 순위표', len(elo_tables), '개')

def clean_elo_table(table):
    parts = table['Club'].str.extract(r'^(\d+)\s+([A-Z]{3})(.+)$')  # 세계 순위, 팀 코드, 팀 이름
    out = pd.DataFrame({'world_rank': parts[0], 'code': parts[1], 'club': parts[2], 'elo': table['Elo']})
    out = out.dropna(subset=['world_rank'])  # 'Level 1 (20 teams)' 같은 구분 행 제거
    return out.astype({'world_rank': int, 'elo': int}).reset_index(drop=True)

england_elo = next(clean_elo_table(t) for t in elo_tables if t['Club'].str.contains('Arsenal').any())
england_elo.head(20)  # 잉글랜드 1~4부 팀들의 Elo

In [ ]:
world_elo = pd.concat([clean_elo_table(t) for t in elo_tables]).sort_values('world_rank').reset_index(drop=True)
world_elo.head(20)  # 세계 Elo 상위 20팀

### 특정 팀의 최근 경기별 Elo 변화
- 팀 페이지(`clubelo.com/팀이름`)에는 최근 경기마다 상대, 결과, Elo 증감, 새 순위가 표로 나옵니다.

In [ ]:
liverpool_pages = read_tables('http://clubelo.com/Liverpool')
liverpool_recent = next(t for t in liverpool_pages if 'New Elo' in t.columns)
liverpool_recent = liverpool_recent.rename(columns={liverpool_recent.columns[0]: 'Date'})
liverpool_recent[['Date', 'H/A', 'Opponent', 'FT', 'Elo +/-', 'New Elo', 'New Rank']]

## [Transfermarkt](https://www.transfermarkt.com/)
- 선수 시장가치, 이적, 계약, 스쿼드 정보로 유명한 사이트. 리그 순위표도 제공합니다.
- API가 없으므로 ClubElo와 같은 방식으로 웹페이지 표를 읽습니다. 브라우저 User-Agent 헤더가 없으면 403으로 차단됩니다.
- 요청을 짧은 시간에 너무 많이 보내면 차단될 수 있으니 페이지 몇 개만 읽습니다.

### 리그 순위표

In [ ]:
TM = 'https://www.transfermarkt.com'

tm_tables = read_tables(f'{TM}/premier-league/tabelle/wettbewerb/GB1/saison_id/2026')
tm_table = next(t for t in tm_tables if 'Pts' in t.columns)
tm_table = tm_table.dropna(axis=1, how='all')  # 로고 이미지 열은 값이 없어서 비어 있음
tm_table.columns = ['Pos', 'Club', 'MP', 'W', 'D', 'L', 'Goals', 'GD', 'Pts']
tm_table

### 구단별 시장가치
- `€1.43bn`, `€57.39m` 같은 문자열은 그대로는 계산할 수 없으니 숫자(백만 유로 단위)로 바꾸는 함수를 만듭니다.

In [ ]:
def money_to_million_euro(text):
    """'€1.43bn' -> 1430.0, '€57.39m' -> 57.39, '€500k' -> 0.5 (단위: 백만 유로)"""
    match = re.match(r'€([\d.]+)(bn|m|k)', str(text))
    if match is None:
        return None
    value, unit = float(match.group(1)), match.group(2)
    return value * {'bn': 1000, 'm': 1, 'k': 0.001}[unit]

club_pages = read_tables(f'{TM}/premier-league/startseite/wettbewerb/GB1')
club_values = next(t for t in club_pages if 'Total market value' in t.columns)
club_values = club_values.dropna(axis=1, how='all').iloc[:, :6]
club_values.columns = ['Club', 'Squad', 'Avg age', 'Foreigners', 'Avg market value', 'Total market value']
club_values = club_values[club_values['Club'].notna()].copy()  # 마지막 합계 행 제거
club_values['Total (€m)'] = club_values['Total market value'].map(money_to_million_euro)
club_values

### 선수 시장가치 상위 선수
- 이 페이지는 선수 이름·포지션이 한 칸에 겹쳐 있고 소속 구단은 로고 이미지라서 `read_html`만으로는 깔끔하게 안 나옵니다. 이럴 때는 `BeautifulSoup`으로 HTML 구조를 직접 읽습니다.
- 브라우저의 개발자 도구(F12)로 표의 `<tr>`, `<td>` 구조를 확인한 뒤 필요한 칸을 골라내는 방식입니다.

In [ ]:
soup = BeautifulSoup(get_html(f'{TM}/premier-league/marktwerte/wettbewerb/GB1'), 'html.parser')

rows = []
for tr in soup.select('table.items > tbody > tr'):
    tds = tr.find_all('td', recursive=False)
    if len(tds) != 6:  # 선수 행이 아닌 것(광고, 구분선 등)은 건너뜀
        continue
    rows.append({
        'player': tds[1].find('img')['alt'],
        'position': tds[1].select_one('table tr:nth-of-type(2) td').get_text(strip=True),
        'nationality': tds[2].find('img')['title'],
        'age': int(tds[3].get_text(strip=True)),
        'club': tds[4].find('img')['alt'],
        'market_value': tds[5].get_text(strip=True),
    })

player_values = pd.DataFrame(rows)
player_values['value (€m)'] = player_values['market_value'].map(money_to_million_euro)
player_values

In [ ]:
player_values.groupby('club')['value (€m)'].agg(['count', 'sum']).sort_values('sum', ascending=False)  # 상위 선수를 많이 보유한 구단

### 특정 구단의 스쿼드
- 구단 페이지의 스쿼드 표는 선수 한 명이 세 줄(이름+포지션, 이름, 포지션)로 나뉘어 있어 `#`(등번호)가 있는 줄만 남기고 이름과 포지션을 아래 두 줄에서 가져옵니다.

In [ ]:
squad_pages = read_tables(f'{TM}/liverpool-fc/kader/verein/31/saison_id/2026/plus/1')
squad_raw = next(t for t in squad_pages if 'Market value' in t.columns)

squad = squad_raw[squad_raw['#'].notna()].copy()
squad['Player'] = squad_raw['Player'].shift(-1)[squad.index].values    # 바로 아래 줄: 이름
squad['Position'] = squad_raw['Player'].shift(-2)[squad.index].values  # 그 아래 줄: 포지션
squad = squad.dropna(axis=1, how='all')
squad['Value (€m)'] = squad['Market value'].map(money_to_million_euro)
squad[['#', 'Player', 'Position', 'Date of birth/Age', 'Height', 'Foot', 'Joined', 'Contract', 'Market value', 'Value (€m)']].reset_index(drop=True)

In [ ]:
squad.groupby('Position')['Value (€m)'].agg(['count', 'sum', 'mean']).sort_values('sum', ascending=False)  # 포지션별 시장가치

## [Understat](https://understat.com/)

- 자체 xG 모델과 여러 기록이 있는 통계 사이트
- [soccerdata](https://soccerdata.readthedocs.io/en/latest/datasources/Understat.html) 라이브러리의 Understat 리더를 사용합니다. 사이트 구조가 바뀌면 라이브러리도 같이 업데이트해야 하므로, 오류가 나면 먼저 `pip install -U soccerdata`로 최신 버전인지 확인하세요.

In [ ]:
us = sd.Understat(leagues="ENG-Premier League", seasons="2026/2027")

### 경기 일정과 링크 불러오기

In [ ]:
schedule = us.read_schedule().reset_index()
us_match_links = schedule['url'].tolist()
schedule[['date', 'home_team', 'away_team', 'home_goals', 'away_goals', 'home_xg', 'away_xg', 'url']].head()

### 팀 순위 보기

In [ ]:
team_match = us.read_team_match_stats().reset_index()

# 경기 단위 기록을 팀 단위(홈/원정)로 풀어서 순위표 만들기
cols = ['team', 'points', 'goals_for', 'goals_against', 'xg', 'xga']
home_rows = team_match[['home_team', 'home_points', 'home_goals', 'away_goals', 'home_xg', 'away_xg']].set_axis(cols, axis=1).assign(venue='Home')
away_rows = team_match[['away_team', 'away_points', 'away_goals', 'home_goals', 'away_xg', 'home_xg']].set_axis(cols, axis=1).assign(venue='Away')
team_rows = pd.concat([home_rows, away_rows], ignore_index=True)

def league_table(df):
    table = df.groupby('team').agg(
        M=('points', 'size'), Pts=('points', 'sum'),
        GF=('goals_for', 'sum'), GA=('goals_against', 'sum'),
        xG=('xg', 'sum'), xGA=('xga', 'sum'),
    )
    table['GD'] = table['GF'] - table['GA']
    return table.sort_values(['Pts', 'GD', 'GF'], ascending=False)

tables = [league_table(team_rows),
          league_table(team_rows[team_rows['venue'] == 'Home']),
          league_table(team_rows[team_rows['venue'] == 'Away'])]

In [ ]:
tables[0] # 전체 순위, 홈 경기 순위, 원정 경기 순위

### 팀 내 순위 보기

In [ ]:
player_stats = us.read_player_season_stats().reset_index()

In [ ]:
player_stats['team'].unique()

In [ ]:
everton = player_stats[player_stats['team'] == 'Everton']
everton[['player', 'position', 'matches', 'minutes', 'goals', 'xg', 'assists', 'xa', 'shots', 'key_passes']].sort_values('xg', ascending=False).head()

### 경기 정보 보기

In [ ]:
first_match = schedule.iloc[0]
us_match = us.read_player_match_stats(match_id=int(first_match['game_id'])).reset_index()

In [ ]:
us_match['team'].unique() # home away 구분

In [ ]:
us_match_df = us_match[us_match['team'] == first_match['home_team']] # 첫번째 경기의 홈 팀 정보

In [ ]:
us_match_df.head()

### 여러 경기의 슈팅 데이터 보기

In [ ]:
match_ids = schedule['game_id'].iloc[:5].astype(int).tolist() # 처음 5경기
us_shots = us.read_shot_events(match_id=match_ids)

In [ ]:
us_shots.reset_index()['game'].value_counts()

In [ ]:
us_shots.head()

### 시즌 누적 데이터 불러오기

In [ ]:
player_stats.sort_values('xg', ascending=False).head(10) # 시즌 xG 상위 10명

### 특정 팀의 경기 데이터 불러오기

In [ ]:
team_name = 'Aston Villa'
us_team_data = team_match[(team_match['home_team'] == team_name) | (team_match['away_team'] == team_name)]
us_team_data[['date', 'home_team', 'away_team', 'home_goals', 'away_goals', 'home_xg', 'away_xg', 'home_ppda', 'away_ppda']].head() # 특정 팀(aston villa)의 경기 기록

### [SoccerData](https://soccerdata.readthedocs.io/en/latest/)

In [33]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_context("notebook")
sns.set_style("whitegrid")

#### Home team advantage in the Italian Serie A

In [ ]:
# We all know sports teams have an advantage when playing at home. Here’s a look at home team advantage for 5 years of the Serie A.
try:
    seriea_hist = sd.MatchHistory("ITA-Serie A", range(2018, 2023))
    games = seriea_hist.read_games()
except Exception as e:
    # football-data.co.uk 서버가 응답하지 않으면 아래 홈 어드밴티지 셀들은 건너뜁니다.
    games = None
    print('football-data.co.uk에서 데이터를 받지 못했습니다. 잠시 후 다시 시도해 보세요.', repr(e))
games.sample(5) if games is not None else None

In [37]:
def home_away_results(games: pd.DataFrame):
    """Returns aggregated home/away results per team"""
    res = pd.melt(
        games.reset_index(),
        id_vars=["date", "FTR"],
        value_name="team",
        var_name="is_home",
        value_vars=["home_team", "away_team"],
    )

    res.is_home = res.is_home.replace(["home_team", "away_team"], ["Home", "Away"])
    res["win"] = res["lose"] = res["draw"] = 0
    res.loc[(res["is_home"] == "Home") & (res["FTR"] == "H"), "win"] = 1
    res.loc[(res["is_home"] == "Away") & (res["FTR"] == "A"), "win"] = 1
    res.loc[(res["is_home"] == "Home") & (res["FTR"] == "A"), "lose"] = 1
    res.loc[(res["is_home"] == "Away") & (res["FTR"] == "H"), "lose"] = 1
    res.loc[res["FTR"] == "D", "draw"] = 1

    groups = res.groupby(["team", "is_home"])
    win = groups.win.agg(["sum", "mean"]).rename(columns={"sum": "n_win", "mean": "win_pct"})
    loss = groups.lose.agg(["sum", "mean"]).rename(columns={"sum": "n_lose", "mean": "lose_pct"})
    draw = groups.draw.agg(["sum", "mean"]).rename(columns={"sum": "n_draw", "mean": "draw_pct"})

    res = pd.concat([win, loss, draw], axis=1)
    return res

In [ ]:
results = home_away_results(games) if games is not None else None
results.head(6) if results is not None else None

In [ ]:
# The overall picture shows most teams have a clear advantage at home:
if results is not None:
    g = sns.FacetGrid(results.reset_index(), hue="team", palette="Set2", height=6, aspect=0.5)
    g.map(sns.pointplot, "is_home", "win_pct", order=["Away", "Home"])
    g.set_axis_labels("", "win %");

In [ ]:
# But there are a few exceptions
if results is not None:
    g = sns.FacetGrid(results.reset_index(), col="team", col_wrap=5)
    g.map(sns.pointplot, "is_home", "win_pct", order=["Away", "Home"])
    g.set_axis_labels("", "win %");